In [3]:
import pandas as pd
import openpyxl
import numpy as np
import seaborn as sns
import matplotlib as plt
import plotly.express as px
import plotly.io as pio
from pathlib import Path

In [4]:
#Cargo fichero
df_novaconnect  = pd.read_excel('/Users/rosamariasierraalmeria/Documents/GitHub/contact-center-workforce-prediction/data/NovaConnect_Dataset_V1.xlsx', sheet_name= None)

#Creo bucle para cargar todas las pestañas del archivo.
for nombre_hoja, df in df_novaconnect.items():
    print(nombre_hoja, df.shape)

contactos (189679, 9)
planificacion (10965, 6)
calendario (731, 9)
campanias (3, 5)


In [5]:
#Convierto cada pestaña en una variable para poder trabajar con ellas
df_contactos = df_novaconnect["contactos"]
df_agentes = df_novaconnect["planificacion"]
df_calendario = df_novaconnect["calendario"]
df_campañas = df_novaconnect["campanias"]

In [54]:
df_contactos.info()

<class 'pandas.DataFrame'>
RangeIndex: 189679 entries, 0 to 189678
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   id_contacto        189679 non-null  int64
 1   fecha              189679 non-null  str  
 2   hora_inicio        189679 non-null  str  
 3   canal              189679 non-null  str  
 4   departamento       189679 non-null  str  
 5   duracion_segundos  189679 non-null  int64
 6   resuelto           189679 non-null  str  
 7   SLA_cumplido       189679 non-null  str  
 8   campania           189679 non-null  str  
dtypes: int64(2), str(7)
memory usage: 13.0 MB


In [55]:
df_contactos.head()

,id_contacto,fecha,hora_inicio,canal,departamento,duracion_segundos,resuelto,SLA_cumplido,campania
0,1,2024-01-01,08:50,Teléfono,ATC,330,Sí,Sí,No
1,2,2024-01-01,08:01,Chat,ATC,393,Sí,Sí,No
2,3,2024-01-01,08:59,Teléfono,ATC,374,Sí,Sí,No
3,4,2024-01-01,08:26,Chat,ATC,373,Sí,Sí,No
4,5,2024-01-01,08:43,Teléfono,ATC,355,Sí,Sí,No


In [56]:
#Modifico el formato de 'hora_inicio' de str a horas.
df_contactos['hora_inicio'] = pd.to_datetime(df_contactos['hora_inicio'], format='%H:%M').dt.hour

In [57]:
#Modifico el formato de 'fecha' de str a datetime64.
df_contactos['fecha'] = pd.to_datetime(df_contactos['fecha'])

In [58]:
#Convierto el formato de las columnas 'resuelto' y 'SLA_cumplido' a True/False para que sean booleanos.
#.map() ecorre la columna valor a valor y sustituye cada texto por lo que le digas en el diccionario.
df_contactos['resuelto'] = df_contactos['resuelto'].map({'Sí': True, 'No': False})
df_contactos['SLA_cumplido'] = df_contactos['SLA_cumplido'].map({'Sí': True, 'No': False})

In [59]:
df_contactos['campania'] = df_contactos['campania'].astype('category')

In [60]:
#Compruebo el resultado de los cambios de formato.
df_contactos.info()

<class 'pandas.DataFrame'>
RangeIndex: 189679 entries, 0 to 189678
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   id_contacto        189679 non-null  int64         
 1   fecha              189679 non-null  datetime64[us]
 2   hora_inicio        189679 non-null  int32         
 3   canal              189679 non-null  str           
 4   departamento       189679 non-null  str           
 5   duracion_segundos  189679 non-null  int64         
 6   resuelto           189679 non-null  bool          
 7   SLA_cumplido       189679 non-null  bool          
 8   campania           189679 non-null  category      
dtypes: bool(2), category(1), datetime64[us](1), int32(1), int64(2), str(2)
memory usage: 8.5 MB


In [61]:
df_agentes.info()

<class 'pandas.DataFrame'>
RangeIndex: 10965 entries, 0 to 10964
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   fecha                10965 non-null  str  
 1   franja_horaria       10965 non-null  str  
 2   agentes_programados  10965 non-null  int64
 3   agentes_presentes    10965 non-null  int64
 4   ausencias            10965 non-null  int64
 5   vacaciones           10965 non-null  int64
dtypes: int64(4), str(2)
memory usage: 514.1 KB


In [62]:
df_agentes.head()

,fecha,franja_horaria,agentes_programados,agentes_presentes,ausencias,vacaciones
0,2024-01-01,08:00-09:00,29,27,2,6
1,2024-01-01,09:00-10:00,41,38,3,6
2,2024-01-01,10:00-11:00,26,24,2,6
3,2024-01-01,11:00-12:00,38,37,1,6
4,2024-01-01,12:00-13:00,38,35,3,6


In [63]:
#Convierto columna 'fecha' a datetime.
df_agentes['fecha'] = pd.to_datetime(df_agentes['fecha'])

In [64]:
#No convierto columna 'franja_horaria' a datetime
#Compruebo valores únicos para decidir si el dtype de la columna lo dejo como está o lo modifico.
print(df_agentes['franja_horaria'].nunique())  
print(len(df_agentes['franja_horaria'])) 

15
10965


In [65]:
#Convierto a category ya que la columna tiene pocos valores repetidos.
df_agentes['franja_horaria'] = df_agentes['franja_horaria'].astype('category')

In [66]:
#Compruebo el resultado de los cambios de formato.
df_agentes.info()

<class 'pandas.DataFrame'>
RangeIndex: 10965 entries, 0 to 10964
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   fecha                10965 non-null  datetime64[us]
 1   franja_horaria       10965 non-null  category      
 2   agentes_programados  10965 non-null  int64         
 3   agentes_presentes    10965 non-null  int64         
 4   ausencias            10965 non-null  int64         
 5   vacaciones           10965 non-null  int64         
dtypes: category(1), datetime64[us](1), int64(4)
memory usage: 439.3 KB


In [67]:
df_calendario.info()

<class 'pandas.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   fecha       731 non-null    str  
 1   anio        731 non-null    int64
 2   mes         731 non-null    int64
 3   trimestre   731 non-null    int64
 4   semana      731 non-null    int64
 5   dia_semana  731 non-null    str  
 6   fin_semana  731 non-null    str  
 7   festivo     731 non-null    str  
 8   campania    84 non-null     str  
dtypes: int64(4), str(5)
memory usage: 51.5 KB


In [6]:
df_calendario.head()

,fecha,anio,mes,trimestre,semana,dia_semana,fin_semana,festivo,campania
0,2024-01-01,2024,1,1,1,Lunes,No,Sí,NaN
1,2024-01-02,2024,1,1,1,Martes,No,No,NaN
2,2024-01-03,2024,1,1,1,Miércoles,No,No,NaN
3,2024-01-04,2024,1,1,1,Jueves,No,No,NaN
4,2024-01-05,2024,1,1,1,Viernes,No,No,NaN


In [9]:
#Convierto columna 'fecha' a datetime.
df_calendario['fecha'] = pd.to_datetime(df_calendario['fecha'])

In [10]:
#Convierto columna 'anio' a category.
df_calendario['anio'] = df_calendario['anio'].astype('category')

In [11]:
#Convierto columna 'mes' a category.
df_calendario['mes'] = df_calendario['mes'].astype('category')

In [12]:
#Convierto columna 'trimestre' a category.
df_calendario['trimestre'] = df_calendario['trimestre'].astype('category')

In [13]:
#Convierto columna 'semana' a category.
df_calendario['semana'] = df_calendario['semana'].astype('category')

In [14]:
#Convierto columna 'dia_semana' a category.
df_calendario['dia_semana'] = df_calendario['dia_semana'].astype('category')

In [7]:
#Convierto el formato de las columnas 'fin_semana' a True/False para que sean booleanos.
df_calendario['fin_semana'] = df_calendario['fin_semana'].map({'Sí': True, 'No': False})

In [8]:
#Convierto el formato de las columnas 'festivo' a True/False para que sean booleanos.
df_calendario['festivo'] = df_calendario['festivo'].map({'Sí': True, 'No': False})

In [16]:
df_calendario['campania'] = df_calendario['campania'].astype('category')

In [17]:
#Compruebo el resultado de los cambios de formato.
df_calendario.info()

<class 'pandas.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       731 non-null    datetime64[us]
 1   anio        731 non-null    category      
 2   mes         731 non-null    category      
 3   trimestre   731 non-null    category      
 4   semana      731 non-null    category      
 5   dia_semana  731 non-null    category      
 6   fin_semana  731 non-null    bool          
 7   festivo     731 non-null    bool          
 8   campania    84 non-null     category      
dtypes: bool(2), category(6), datetime64[us](1)
memory usage: 12.2 KB


In [35]:
df_campañas.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   campania            3 non-null      str  
 1   inicio              3 non-null      str  
 2   fin                 3 non-null      str  
 3   incremento_volumen  3 non-null      str  
 4   incremento_AHT      3 non-null      str  
dtypes: str(5)
memory usage: 252.0 bytes


In [36]:
df_campañas.head()

,campania,inicio,fin,incremento_volumen,incremento_AHT
0,Rebajas,07/01,20/01,15%,8%
1,Black Friday,20/11,30/11,35%,8%
2,Navidad,15/12,31/12,20%,8%
